In [1]:
import pandas as pd
import json
import numpy as np
import re
from sklearn.neural_network import MLPRegressor

In [18]:
climate = {}
with open('/work/wildfirerisk/europe_dataset/climate_data.json', 'r') as f:
    md = json.load(f)
    for k in md:
        climate[k] = md[k]
with open('/work/wildfirerisk/europe_dataset/climate_data_2.json', 'r') as f:
    md = json.load(f)
    for k in md:
        climate[k] = md[k]
with open('/work/wildfirerisk/europe_dataset/climate_data_3.json', 'r') as f:
    md = json.load(f)
    for k in md:
        climate[k] = md[k]
with open('/work/wildfirerisk/large_dataset/climate_data.json', 'r') as f:
    md = json.load(f)
    for k in md:
        climate[k] = md[k]
with open('/work/wildfirerisk/small_dataset/climate_data.json', 'r') as f:
    md = json.load(f)
    for k in md:
        climate[k] = md[k]
len(climate)

23930

In [3]:
from pathlib import Path
_COORD_RE = re.compile(r"lon(?P<lon>-?[0-9]+(?:\.[0-9]+)?)_lat(?P<lat>-?[0-9]+(?:\.[0-9]+)?)")

def parse_lat_lon_from_name(path):
    m = _COORD_RE.search(Path(path).name)
    if not m:
        return None
    lon = float(m.group('lon'))
    lat = float(m.group('lat'))
    return f"{lat}_{lon}"

def dict_to_vector(data):
    def flatten(d):
        items = []
        for key in sorted(d.keys()):
            value = d[key]
            if isinstance(value, dict):
                items.extend(flatten(value))
            else:
                items.append(value)
        return items

    return flatten(data)

def vectorize(raw, ref_for_norm):
    keys = []
    mats, mats_for_norm = [], []
    for k, v in raw.items():
        vec = np.asarray(dict_to_vector(v), dtype=np.float32)
        mats.append(vec); keys.append(k)
        if k in ref_for_norm:
            mats_for_norm.append(vec)
    mats = np.stack(mats, axis=0)
    mats_for_norm = np.stack(mats_for_norm, axis=0)
    assert mats.shape[1] == 60, f"cond_dim mismatch: got {mats.shape[1]}, expected 60"
    mu = mats_for_norm.mean(0, keepdims=True)
    sd = mats_for_norm.std(0, keepdims=True) + 1e-6
    mats = (mats - mu) / sd
    vec_map = {k: mats[i] for i, k in enumerate(keys)}
    return vec_map

In [4]:
y = pd.read_json('/work/wildfirerisk/large_dataset/risk_rasters/tile_extrametadata.json')
y['lat_lon'] = y.tile_file.apply(parse_lat_lon_from_name)
y_train = y[y['subset']=='train']
y_train = pd.Series(y_train['mean_normalised_risk'].values, index=y_train['lat_lon'])
x = vectorize(climate, y_train.index)
x = pd.DataFrame(x).T

In [106]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = MLPRegressor(hidden_layer_sizes=(16)).fit(x.loc[y_train.index], y_train)
y_val = y[y['subset']=='val']
y_val = pd.Series(y_val['mean_normalised_risk'].values, index=y_val['lat_lon'])
y_pred = model.predict(x.loc[y_val.index])
print(mean_absolute_error(y_val, y_pred), mean_squared_error(y_val, y_pred))

0.12901416039092148 0.02975195310340546


In [107]:
import pickle
with open('/work/wildfirerisk/climate_mlp.pkl', 'wb') as f:
    pickle.dump(model, f)

In [12]:
import pickle
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

with open('/work/wildfirerisk/climate_mlp.pkl', 'rb') as f:
    model = pickle.load(f)
y_val = y[y['subset']=='val']
y_val = pd.Series(y_val['mean_normalised_risk'].values, index=y_val['lat_lon'])
y_pred = model.predict(x.loc[y_val.index])
print(mean_absolute_error(y_val, y_pred), mean_squared_error(y_val, y_pred))

0.12901416039092148 0.02975195310340546


In [108]:
template = pd.read_json("/work/wildfirerisk/vlm_cot_predictions.json").T

climate_pred, q_climate_pred, random = {}, {}, {}
for lat_lon, row in template.iterrows():
    pred = model.predict(x.loc[[lat_lon]])[0]
    climate_pred[lat_lon] =  {'answer': min(float(pred)*10, 9), 'ground_truth': row.ground_truth, 'img_path': row.img_path}
    q_climate_pred[lat_lon] =  {'answer': min(int(np.floor(pred*10)), 9), 'ground_truth': row.ground_truth, 'img_path': row.img_path}
    random[lat_lon] = {'answer': np.random.randint(10), 'ground_truth': row.ground_truth, 'img_path': row.img_path}

with open("/work/wildfirerisk/climate_predictions.json", 'w') as f:
    json.dump(climate_pred, f)

with open("/work/wildfirerisk/climate-answers_predictions.json", 'w') as f:
    json.dump(q_climate_pred, f)

with open("/work/wildfirerisk/random_predictions.json", 'w') as f:
    json.dump(random, f)

In [20]:
template = pd.read_json("/work/wildfirerisk/vlm_cot_output_small_dataset.json").T

climate_pred = {}
for lat_lon, row in template.iterrows():
    pred = model.predict(x.loc[[lat_lon]])[0]
    climate_pred[lat_lon] =  {'answer': min(float(pred)*10, 9), 'ground_truth': row.ground_truth, 'img_path': row.img_path}

with open("/work/wildfirerisk/climate_predictions_small_dataset.json", 'w') as f:
    json.dump(climate_pred, f)

In [ ]:
import pandas as pd
import json
from sklearn.preprocessing import quantile_transform

template = pd.read_json("/work/wildfirerisk/gpt_predictions.json").T
fwis = {}
for lat_lon, row in template.iterrows():
    img_path = row.img_path
    if 'europe' not in img_path:
        continue
    lat, lon = map(float, lat_lon.split('_'))
    if 'no_fire' in img_path:
        fwi = np.mean([get_fwi_at_point(lat,lon,i) for i in range(d.shape[0])])
    else:
        year = int(img_path.split('/')[-2])-2015
        year = min(year, 9)
        fwi = get_fwi_at_point(lat,lon,year)
    fwis[lat_lon] = fwi

fwis = pd.Series(fwis)
fwis = pd.Series(quantile_transform(fwis.values.reshape(-1,1))[:,0], index=fwis.index).to_dict()

predictions, answers = {}, {}
for lat_lon, row in template.iterrows():
    if 'europe' not in row.img_path:
        continue
    fwi = fwis[lat_lon]
    answers[lat_lon] =  {'answer': min(float(fwi*10), 9), 'ground_truth': row.ground_truth, 'img_path': row.img_path}
    predictions[lat_lon] = {'answer': int(min(np.floor(fwi*10), 9)), 'ground_truth': row.ground_truth, 'img_path': row.img_path}

with open("/work/wildfirerisk/fwi-mean-jjas_predictions_normed.json", 'w') as f:
    json.dump(predictions, f)

with open("/work/wildfirerisk/fwi-answers_predictions_normed.json", 'w') as f:
    json.dump(answers, f)